# Lab Assignment 2

In [1]:
pip install annoy faiss-cpu hnswlib scikit-learn pandas numpy kagglehub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 26.0 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 154.6 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=521282 sha256=3ea80e9de31310ac710ea927d3b2b9b310d074451bd090fc1e17426fb0e29f9f
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2683501 sha256=9544f4f34e11743f88902d5d62c89ce70b99ace4d0aee55cf2c65928949c2a73
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f

In [2]:
import os
import glob
import time
import numpy as np
import pandas as pd
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

In [3]:
import kagglehub
path = kagglehub.dataset_download("bwandowando/spotify-songs-with-attributes-and-lyrics")
csv_path = os.path.join(path, 'songs_with_attributes_and_lyrics.csv')

if not os.path.exists(csv_path) and os.path.exists('spotify_songs.csv'):
    csv_path = 'spotify_songs.csv'

df = pd.read_csv(csv_path)

100%|██████████| 894M/894M [00:50<00:00, 18.5MB/s]

Extracting files...


In [4]:
features = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]

# Remove missing values
df = df.dropna(subset=features).reset_index(drop=True)
print(f"Loaded {len(df):,} valid tracks with {len(features)} numerical features.")

# Feature standardization
scaler = StandardScaler()
X_scaled = np.ascontiguousarray(scaler.fit_transform(df[features].values), dtype=np.float32)

n_samples, dim = X_scaled.shape
k = 10  # number of nearest neighbors

query_idx = 0
query_vector = X_scaled[query_idx].reshape(1, -1)

# Helper to display track metadata if available in the CSV
meta_cols = [c for c in ['name', 'track_name', 'artists', 'artist_name', 'album_name'] if c in df.columns]
if meta_cols:
    print("\nQuery Track:", df.loc[query_idx, meta_cols].to_dict())

# 2. Exact Nearest Neighbors (Scikit-Learn)

t0 = time.perf_counter()
exact_nn = NearestNeighbors(n_neighbors=k, metric='euclidean', algorithm='brute', n_jobs=-1)
exact_nn.fit(X_scaled)
exact_build_time = time.perf_counter() - t0

t0 = time.perf_counter()
exact_distances, exact_indices = exact_nn.kneighbors(query_vector)
exact_query_time = (time.perf_counter() - t0) * 1000  # ms

gt_set = set(exact_indices[0])

# 3. ANNOY (Approximate Nearest Neighbors)

ann_index = AnnoyIndex(dim, 'euclidean')

t0 = time.perf_counter()
for i in range(n_samples):
    ann_index.add_item(i, X_scaled[i])
ann_index.build(10)  # 10 random projection trees
ann_build_time = time.perf_counter() - t0

t0 = time.perf_counter()
ann_indices, ann_distances = ann_index.get_nns_by_vector(query_vector[0], k, include_distances=True)
ann_query_time = (time.perf_counter() - t0) * 1000  # ms
ann_recall = len(gt_set.intersection(set(ann_indices))) / k

# 4. FAISS (IndexIVFFlat - Approximate Search)

nlist = 100  # number of Voronoi cells
quantizer = faiss.IndexFlatL2(dim)
faiss_index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_L2)

t0 = time.perf_counter()
faiss_index.train(X_scaled)
faiss_index.add(X_scaled)
faiss_build_time = time.perf_counter() - t0

faiss_index.nprobe = 10  # number of cells probed at query time
t0 = time.perf_counter()
faiss_distances, faiss_indices = faiss_index.search(query_vector, k)
faiss_query_time = (time.perf_counter() - t0) * 1000  # ms
faiss_recall = len(gt_set.intersection(set(faiss_indices[0]))) / k

# 5. HNSW (hnswlib)

hnsw_index = hnswlib.Index(space='l2', dim=dim)

t0 = time.perf_counter()
hnsw_index.init_index(max_elements=n_samples, ef_construction=200, M=16)
hnsw_index.add_items(X_scaled)
hnsw_build_time = time.perf_counter() - t0

hnsw_index.set_ef(50)
t0 = time.perf_counter()
hnsw_indices, hnsw_distances = hnsw_index.knn_query(query_vector, k=k)
hnsw_query_time = (time.perf_counter() - t0) * 1000  # ms
hnsw_recall = len(gt_set.intersection(set(hnsw_indices[0]))) / k

# 6. Performance Comparison Table

summary = pd.DataFrame([
    {
        "Algorithm": "Exact NN (Sklearn)",
        "Build Time (s)": f"{exact_build_time:.4f}",
        "Query Time (ms)": f"{exact_query_time:.4f}",
        "Recall@10": "1.00 (Baseline)",
        "Top-5 Neighbor Indices": list(exact_indices[0][:5])
    },
    {
        "Algorithm": "ANNOY (10 trees)",
        "Build Time (s)": f"{ann_build_time:.4f}",
        "Query Time (ms)": f"{ann_query_time:.4f}",
        "Recall@10": f"{ann_recall:.2f}",
        "Top-5 Neighbor Indices": list(ann_indices[:5])
    },
    {
        "Algorithm": "FAISS (IVFFlat)",
        "Build Time (s)": f"{faiss_build_time:.4f}",
        "Query Time (ms)": f"{faiss_query_time:.4f}",
        "Recall@10": f"{faiss_recall:.2f}",
        "Top-5 Neighbor Indices": list(faiss_indices[0][:5])
    },
    {
        "Algorithm": "HNSW (hnswlib)",
        "Build Time (s)": f"{hnsw_build_time:.4f}",
        "Query Time (ms)": f"{hnsw_query_time:.4f}",
        "Recall@10": f"{hnsw_recall:.2f}",
        "Top-5 Neighbor Indices": list(hnsw_indices[0][:5])
    }
])

print("\n=== PERFORMANCE COMPARISON (SPOTIFY DATASET) ===")
print(summary.to_markdown(index=False))

# 7. Inspect Nearest Tracks Found

if meta_cols:
    print("\n=== TOP 10 NEAREST TRACKS (EXACT NN vs HNSW) ===")
    comparison_tracks = pd.DataFrame({
        "Rank": range(1, k + 1),
        "Exact NN Track": df.loc[exact_indices[0], meta_cols[0]].values,
        "Exact Dist": np.round(exact_distances[0], 4),
        "HNSW Track": df.loc[hnsw_indices[0], meta_cols[0]].values,
        "HNSW Dist (Sq L2)": np.round(hnsw_distances[0], 4),
        "ANNOY Track": df.loc[ann_indices, meta_cols[0]].values,
        "FAISS Track": df.loc[faiss_indices[0], meta_cols[0]].values,
    })
    print(comparison_tracks.to_markdown(index=False))

Loaded 955,320 valid tracks with 9 numerical features.

Query Track: {'name': '!', 'artists': 'HELLYEAH', 'album_name': 'UNDEN!ABLE'}

=== PERFORMANCE COMPARISON (SPOTIFY DATASET) ===
| Algorithm          |   Build Time (s) |   Query Time (ms) | Recall@10       | Top-5 Neighbor Indices                                                                     |
|:-------------------|-----------------:|------------------:|:----------------|:-------------------------------------------------------------------------------------------|
| Exact NN (Sklearn) |           0.0035 |           36.6301 | 1.00 (Baseline) | [np.int64(0), np.int64(394553), np.int64(764272), np.int64(837727), np.int64(749223)]      |
| ANNOY (10 trees)   |           4.1562 |            0.2557 | 0.70            | [0, 394553, 764272, 837727, 61511]                                                         |
| FAISS (IVFFlat)    |           0.0923 |            1.2857 | 1.00            | [np.int64(0), np.int64(394553), np.int64(764